<a href="https://colab.research.google.com/github/gayatrishekharcs/AI_LAB/blob/main/Grey_Wolf_Optimizer_Algorithm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# --- STEP 0: PROBLEM CONFIGURATION & PHYSICS SIMULATION ---
NUM_UNITS = 4          # Number of adjustable cooling units
T_MIN = 18.0           # Minimum safe server operating temperature (°C)
T_MAX = 27.0           # Maximum safe server operating temperature (°C)

# Boundaries for the cooling units' control parameters
BOUNDS_AIRFLOW = (1.0, 5.0)    # min and max airflow rate (m^3/s)
BOUNDS_TEMP = (15.0, 25.0)     # min and max cooling air supply temp (°C)

# Combine boundaries into upper and lower bounds arrays for the algorithm
# Configuration Vector structure: [airflow_1, ..., airflow_N, temp_1, ..., temp_N]
lb = np.array([BOUNDS_AIRFLOW[0]] * NUM_UNITS + [BOUNDS_TEMP[0]] * NUM_UNITS)
ub = np.array([BOUNDS_AIRFLOW[1]] * NUM_UNITS + [BOUNDS_TEMP[1]] * NUM_UNITS)
dim = 2 * NUM_UNITS

def evaluate_cooling_fitness(config):
    """
    Simulates data center thermal behavior and calculates fitness score.
    Returns: (Fitness Score [Energy + Penalty], Pure Cooling Energy Consumption)
    """
    airflow = config[:NUM_UNITS]
    cooling_temp = config[NUM_UNITS:]

    # 1. Calculate Cooling Energy Consumption
    # Fan power scales cubically with airflow speed; chiller energy increases as supply temp drops.
    fan_energy = np.sum(45.0 * (airflow ** 3))
    chiller_energy = np.sum(120.0 * (26.0 - cooling_temp))
    total_energy = fan_energy + chiller_energy

    # 2. Simulate Server Heat Profiles
    # Server heat loads (kW) distributed across IT zones
    server_heat_loads = np.array([35.0, 42.0, 38.0, 45.0])

    # Physics approximation: Server Temp = Cooling Temp + Heat Load / (Airflow Capacity)
    server_temps = cooling_temp + (server_heat_loads / (airflow + 0.1))

    # 3. Calculate Boundaries Penalty for Overheating or Under-cooling
    penalty = 0.0
    for t in server_temps:
        if t > T_MAX:
            penalty += 50000.0 * (t - T_MAX) ** 2  # Severe penalty for overheating
        elif t < T_MIN:
            penalty += 5000.0 * (T_MIN - t) ** 2   # Minor penalty for over-cooling

    fitness_score = total_energy + penalty
    return fitness_score, total_energy

# --- MAIN GWO OPTIMIZATION ALGORITHM ---
def grey_wolf_optimizer(num_wolves=30, max_iter=100):
    # 1. Initialize wolves with random positions inside allowable physical bounds
    X = np.random.uniform(lb, ub, size=(num_wolves, dim))

    # Initialize positions and fitness for hierarchy leaders
    alpha_pos = np.zeros(dim)
    alpha_score = float("inf")

    beta_pos = np.zeros(dim)
    beta_score = float("inf")

    delta_pos = np.zeros(dim)
    delta_score = float("inf")

    # Tracking pure energy at optimal configurations
    best_pure_energy = float("inf")

    # 4. Main Optimization Loop
    for iteration in range(max_iter):

        # 2 & 3. Evaluate fitness and identify Alpha, Beta, Delta
        for i in range(num_wolves):
            # Clip wolf positions to physical boundary thresholds before assessment
            X[i] = np.clip(X[i], lb, ub)

            fitness, pure_energy = evaluate_cooling_fitness(X[i])

            # Update Alpha (The best option found)
            if fitness < alpha_score:
                delta_score = beta_score
                delta_pos = beta_pos.copy()
                beta_score = alpha_score
                beta_pos = alpha_pos.copy()
                alpha_score = fitness
                alpha_pos = X[i].copy()
                best_pure_energy = pure_energy

            # Update Beta (Second best option)
            elif fitness < beta_score:
                delta_score = beta_score
                delta_pos = beta_pos.copy()
                beta_score = fitness
                beta_pos = X[i].copy()

            # Update Delta (Third best option)
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = X[i].copy()

        # Linearly decrease parameter 'a' from 2 down to 0
        a = 2.0 - iteration * (2.0 / max_iter)

        # Update Position of all search agents (wolves)
        for i in range(num_wolves):
            for j in range(dim):

                # Equations relative to Alpha
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - X[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Equations relative to Beta
                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - X[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                # Equations relative to Delta
                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - X[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # Calculate new setting position component
                X[i, j] = (X1 + X2 + X3) / 3.0

    # 6 & 7. Return and Display Optimal Outputs
    optimal_airflow = alpha_pos[:NUM_UNITS]
    optimal_temps = alpha_pos[NUM_UNITS:]

    print("=" * 50)
    print(" OPTIMIZATION RESULTS (GREY WOLF OPTIMIZER) ")
    print("=" * 50)
    print(f"Optimal Airflow Rates (m^3/s): {np.round(optimal_airflow, 2)}")
    print(f"Optimal Cooling Temperatures (°C): {np.round(optimal_temps, 2)}")
    print(f"Minimum Cooling Energy Consumption: {round(best_pure_energy, 2)} kW")
    print("=" * 50)

    return alpha_pos, best_pure_energy

# Execute optimization run
if __name__ == "__main__":
    np.random.seed(42) # Seed for reproducibility
    best_config, min_energy = grey_wolf_optimizer(num_wolves=40, max_iter=150)


 OPTIMIZATION RESULTS (GREY WOLF OPTIMIZER) 
Optimal Airflow Rates (m^3/s): [2.82 3.43 3.11 3.66]
Optimal Cooling Temperatures (°C): [15.   15.08 15.14 15.  ]
Minimum Cooling Energy Consumption: 11634.49 kW
